# Model: Neural Network (MLP) for PERM Visa Denial
**Member:** `<YOUR STUDENT ID>`
**Model:** Multi-Layer Perceptron (`sklearn.neural_network.MLPClassifier`)
**Dataset:** `final_processed.csv.gz` (25 selected, scaled features + `denied` target + `split` column)
**Task:** binary classification: `denied` = 1 (Denied) vs 0 (Certified / Certified-Expired)

**Covers:** model design, implementation, hyperparameter tuning, pipeline steps, metrics, k-fold CV and a comparison of model varieties.

> **How to run in Google Colab:** upload `final_processed.csv.gz` when Block 2 asks for it, then use *Runtime > Run all*.
> Set `FAST_MODE = True` in Block 1 for a quick smoke test (about 10% of the rows). Set it back to `False` for the real run.

---
## 1. Why a Neural Network suits this dataset

**How it works.** An MLP is a stack of layers. Each layer multiplies its inputs by a weight matrix, adds a bias and passes the result through a non-linear activation (here ReLU):
`h1 = ReLU(W1·x + b1)`, `h2 = ReLU(W2·h1 + b2)`, ..., and the last layer squashes the result into a probability with a sigmoid: `P(denied) = 1 / (1 + e^-z)`.
Training minimises the log-loss with the **Adam** optimiser (back-propagation on mini-batches). An **L2 penalty** (`alpha`) keeps the weights small and **early stopping** halts training when a held-out slice stops improving.

**Why it fits *this* problem:**
1. **Non-linear interactions.** Denial risk is unlikely to be a simple weighted sum. For example, the effect of `wage_below_pw` probably depends on `pw_level` and on `pwsrc_*`. Hidden layers can learn such interactions automatically; Logistic Regression cannot.
2. **Inputs are already scaled.** Neural networks train badly on unscaled features. Our 25 features were standardised in the pre-processing stage, so they are ready to use.
3. **Enough data.** About 284k training rows is plenty for a small network (a few thousand weights), so over-fitting is manageable.

**Trade-offs (be honest about these in the report):**
- Less interpretable than Logistic Regression (we use *permutation importance* instead of coefficients).
- `MLPClassifier` has no `class_weight`, so the imbalance is handled by **resampling inside the CV pipeline** and by **threshold tuning**.
- Slower to train, and results vary a little with the random seed.

## 2. Evaluation design

| Decision | Choice | Why |
|---|---|---|
| Train / test | The `split` column already in the file (stratified 80/20) | Every model in the group uses the **same rows**, so results are comparable |
| Validation | **5-fold `StratifiedKFold`** (shuffle, seed 42) on the train rows only | k-fold CV is required; stratified keeps ~6.9% denied in every fold |
| Main metric | **PR-AUC** (`average_precision`) | Measures how well denials are ranked above approvals; robust to the rare positive class |
| Also reported | F1, precision and recall for Denied; ROC-AUC; balanced accuracy; accuracy | F1 balances missed denials vs false alarms; accuracy is only a warning sign |
| Test set | Used **once**, at the very end, for the best variety | Using it to make choices would make the reported score optimistic |

**The accuracy trap:** the dataset is ~93% approved, so a model that *always* predicts "approved" scores ~93% accuracy while catching **zero** denials. Variety V0 below demonstrates exactly this.

### Block 1: Setup
Installs, imports and the shared settings (`SEED`, CV splitter, scoring dictionary).

In [ ]:
# imbalanced-learn provides SMOTE / RandomOverSampler and a Pipeline that only resamples the TRAIN part of each fold
!pip install -q imbalanced-learn

import os, glob, json, time, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (StratifiedKFold, GridSearchCV, cross_validate,
                                     cross_val_predict, train_test_split)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score, precision_score, recall_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve, ConfusionMatrixDisplay, make_scorer)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler

warnings.filterwarnings('ignore')          # MLP prints a ConvergenceWarning when it hits max_iter; we ignore it here

# ---------------- global settings ----------------
FAST_MODE = False          # True = quick smoke test on ~10% of the rows (results are NOT meaningful)
SEED      = 42             # one seed everywhere so the run is reproducible
TARGET    = 'denied'
MAX_ITER  = 15 if FAST_MODE else 60       # upper limit of training epochs for one MLP
FIXED_EPOCHS = 8 if FAST_MODE else 15     # epochs for the resampled varieties (see Block 7)

MODEL_DIR = 'results/models/NeuralNetwork'
os.makedirs(MODEL_DIR, exist_ok=True)

# ---------------- cross-validation + metrics (shared by every variety) ----------------
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)   # stratified keeps the 6.9% denied rate in each fold
SCORING = {
    'pr_auc':       'average_precision',                              # main metric (threshold-free)
    'roc_auc':      'roc_auc',
    'f1':           make_scorer(f1_score, zero_division=0),           # these three use the 0.5 threshold
    'precision':    make_scorer(precision_score, zero_division=0),
    'recall':       make_scorer(recall_score, zero_division=0),
    'balanced_acc': 'balanced_accuracy',
    'accuracy':     'accuracy',
}

### Block 2: Load the dataset
**What:** find `final_processed.csv.gz` (or upload it in Colab) and split it with the existing `split` column.
**Why:** no new `train_test_split`. The split was fixed earlier, so every group member trains and tests on identical rows.

In [ ]:
# ---------------- locate the data file ----------------
def find_data_file():
    """Look for the file in the usual places; if we are in Colab and it is missing, ask for an upload."""
    for folder in ['.', '/content', '/content/drive/MyDrive', '/mnt/user-data/uploads']:
        hits = glob.glob(os.path.join(folder, '*final_processed*'))
        if hits:
            return hits[0]
    try:
        from google.colab import files          # only exists inside Colab
        print('Please choose final_processed.csv.gz ...')
        uploaded = files.upload()
        return next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError('Put final_processed.csv.gz next to this notebook.')

DATA_FILE = find_data_file()
print('Using data file:', DATA_FILE)

# pandas reads .gz directly (compression is inferred from the extension; we force it in case the name was altered)
df = pd.read_csv(DATA_FILE, compression='gzip')

# ---------------- split with the existing column ----------------
train, test = df[df['split'].eq('train')], df[df['split'].eq('test')]
X_train, y_train = train.drop(columns=[TARGET, 'split']), train[TARGET]
X_test,  y_test  = test.drop(columns=[TARGET, 'split']),  test[TARGET]

if FAST_MODE:   # keep ~10% of train and ~25% of test, stratified, just to check the notebook runs end to end
    X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=0.10, stratify=y_train, random_state=SEED)
    X_test,  _, y_test,  _ = train_test_split(X_test,  y_test,  train_size=0.25, stratify=y_test,  random_state=SEED)

print(f'X_train {X_train.shape} | X_test {X_test.shape} | features: {X_train.shape[1]}')
print(f'Missing values: {int(df.isna().sum().sum())}')
print(f'Denied rate - train {y_train.mean():.2%} | test {y_test.mean():.2%}')
print(f'"Always approve" accuracy on test: {1 - y_test.mean():.2%}')

### Block 3: A helper that scores any model with 5-fold CV
**What:** `cross_validate` trains the model 5 times; each time it holds out one fold of the **train** rows and scores on it. We record the mean ± standard deviation of every metric.
**Why:** one function means every variety is scored identically. The standard deviation shows how *stable* a result is across folds.

`make_mlp()` builds our base network so every variety starts from the same settings and only changes what we want to test.

Note: F1, precision and recall use the default 0.5 threshold here; PR-AUC and ROC-AUC are threshold-free.

In [ ]:
def make_mlp(**overrides):
    """Base network. Anything passed in `overrides` replaces the default value."""
    params = dict(
        hidden_layer_sizes=(64, 32),   # two hidden layers: 64 then 32 neurons
        activation='relu',             # non-linearity between layers
        solver='adam',                 # adaptive mini-batch gradient descent
        alpha=1e-4,                    # L2 penalty on the weights (regularisation)
        batch_size=512,                # rows per gradient step
        learning_rate_init=1e-3,       # Adam step size
        max_iter=MAX_ITER,             # maximum number of epochs
        early_stopping=True,           # stop when an internal 10% slice stops improving ...
        validation_fraction=0.1,
        n_iter_no_change=5,            # ... for 5 epochs in a row
        random_state=SEED,
    )
    params.update(overrides)
    return MLPClassifier(**params)

results = []    # one row per variety
fitted  = {}    # name -> (estimator, X_train, X_test), used later for the final model

def evaluate_cv(name, description, estimator, X, y, X_te):
    """5-fold CV on the train rows; stores mean +/- std of every metric in `results`."""
    t = time.time()
    cv = cross_validate(estimator, X, y, cv=CV, scoring=SCORING, n_jobs=-1)
    row = {'variety': name, 'description': description}
    for m in SCORING:
        row[f'{m}_mean'] = cv[f'test_{m}'].mean()
        row[f'{m}_std']  = cv[f'test_{m}'].std()
    row['seconds'] = round(time.time() - t, 1)
    results.append(row)
    fitted[name] = (estimator, X, X_te)
    print(f"{name}: PR-AUC {row['pr_auc_mean']:.3f} +/- {row['pr_auc_std']:.3f} | F1 {row['f1_mean']:.3f} | "
          f"recall {row['recall_mean']:.3f} | precision {row['precision_mean']:.3f} | accuracy {row['accuracy_mean']:.3f} "
          f"({row['seconds']}s)")
    return row

## 3. Model varieties ("different pre-processing and hyperparameter tune-ups")

| Variety | What changes | Question it answers |
|---|---|---|
| **V0** | `DummyClassifier`: always predicts "approved" | What does a useless model score? (the accuracy trap) |
| **V1** | Plain MLP, 2 hidden layers (64, 32), default settings | What happens if we ignore the imbalance? |
| **V2** | `RandomOverSampler` inside the CV pipeline | Does simply duplicating denied rows help? |
| **V3** | `GridSearchCV` over network size × `alpha` | What architecture / regularisation is best? |
| **V4** | SMOTE inside the CV pipeline + best V3 settings | Is synthetic oversampling better than duplication? |
| **V5** | Best V3 settings **without** the `pwsrc_*` columns | Ablation: how much does the "incomplete application" shortcut contribute? |

### Block 4: V0 baseline and V1 plain MLP
**Expect:** V0 has ~93% accuracy but 0 recall, which proves accuracy is meaningless here. V1 learns *something* (PR-AUC well above the 0.069 random level), but because denials are rare it rarely predicts "denied" at the 0.5 threshold, so recall stays low.

In [ ]:
evaluate_cv('V0', 'Dummy: always approve', DummyClassifier(strategy='most_frequent'), X_train, y_train, X_test)
evaluate_cv('V1', 'Plain MLP (64, 32)', make_mlp(), X_train, y_train, X_test)

### Block 5: V2 random oversampling inside a pipeline
**What:** `RandomOverSampler(sampling_strategy=0.5)` duplicates random denied rows until denied rows = 50% of approved rows (a full 1:1 balance would double the training time for little gain).
**Why inside a Pipeline:** the `imblearn` Pipeline resamples **only the training part of each fold**. The validation fold and the test set stay 100% real data. Resampling before splitting would leak copies of validation rows into training.
**Why fixed epochs:** with duplicated rows, MLP's internal early-stopping slice would contain copies of training rows and would stop too late. So for resampled varieties we switch early stopping off and train a fixed number of epochs (`FIXED_EPOCHS`).
**Expect:** recall jumps, precision drops (more false alarms). PR-AUC changes little, because resampling mainly shifts the threshold, not the ranking.

In [ ]:
over_mlp = ImbPipeline([
    ('over', RandomOverSampler(sampling_strategy=0.5, random_state=SEED)),
    ('mlp',  make_mlp(early_stopping=False, max_iter=FIXED_EPOCHS)),
])
evaluate_cv('V2', 'RandomOverSampler (in-fold) + MLP', over_mlp, X_train, y_train, X_test)

### Block 6: V3 hyperparameter tuning with GridSearchCV
**Hyperparameters:**
- **`hidden_layer_sizes`** = the architecture. `(32,)` is one small layer; `(64, 32)` is two layers; `(128, 64, 32)` is a deeper, wider network. More capacity can learn more complex patterns but can also over-fit.
- **`alpha`** = strength of the L2 penalty. Small `alpha` (1e-4) = weak regularisation; larger `alpha` (1e-2) = weights are pulled towards zero, giving a simpler model.

**How:** 3 architectures × 2 `alpha` values = 6 combinations × 5 folds = **30 fits**, scored by PR-AUC on the held-out folds. The best combination is then refit on all train rows.

In [ ]:
param_grid = {
    'hidden_layer_sizes': [(32,), (64, 32), (128, 64, 32)],
    'alpha': [1e-4, 1e-2],
}
grid = GridSearchCV(make_mlp(), param_grid, scoring='average_precision', cv=CV, n_jobs=-1, return_train_score=True)
t = time.time()
grid.fit(X_train, y_train)
print(f'Grid search: {len(grid.cv_results_["params"])} combinations x 5 folds in {time.time() - t:.0f}s')
print('Best parameters:', grid.best_params_, f'| best CV PR-AUC {grid.best_score_:.4f}')

grid_table = pd.DataFrame(grid.cv_results_)[['param_hidden_layer_sizes', 'param_alpha', 'mean_test_score',
                                              'std_test_score', 'mean_train_score', 'rank_test_score']]
grid_table = grid_table.rename(columns={'mean_test_score': 'cv_pr_auc', 'std_test_score': 'cv_std',
                                        'mean_train_score': 'train_pr_auc'}).sort_values('rank_test_score')
grid_table['param_hidden_layer_sizes'] = grid_table['param_hidden_layer_sizes'].astype(str)
grid_table.to_csv(os.path.join(MODEL_DIR, 'grid_search_results.csv'), index=False)
grid_table.round(4)   # a big gap between train_pr_auc and cv_pr_auc means over-fitting

In [ ]:
# Visualise the grid: CV PR-AUC for every architecture, one bar colour per alpha
fig, ax = plt.subplots(figsize=(8, 4.5))
archs = sorted(grid_table['param_hidden_layer_sizes'].unique(), key=lambda s: (s.count(','), s))
width = 0.38
for i, (a, grp) in enumerate(grid_table.groupby('param_alpha')):
    grp = grp.set_index('param_hidden_layer_sizes').loc[archs]
    ax.bar(np.arange(len(archs)) + (i - 0.5) * width, grp['cv_pr_auc'], width, yerr=grp['cv_std'],
           capsize=3, label=f'alpha = {a}')
ax.set_xticks(np.arange(len(archs)), archs)
ax.set_xlabel('hidden_layer_sizes')
ax.set_ylabel('5-fold CV PR-AUC')
ax.set_title('V3 GridSearchCV: PR-AUC by architecture and alpha', fontweight='bold')
ax.legend()
plt.savefig(os.path.join(MODEL_DIR, 'grid_search_pr_auc.png'), dpi=200, bbox_inches='tight')
plt.show()

# Score the best combination with the same helper as every other variety
best = grid.best_params_
evaluate_cv('V3', f"GridSearch best: layers={best['hidden_layer_sizes']}, alpha={best['alpha']}",
            make_mlp(**best), X_train, y_train, X_test)

### Block 7: V4 SMOTE inside a pipeline
**What:** SMOTE creates *synthetic* denied applications by interpolating between real denied neighbours (up to 50% of the approved count, as in V2).
**Why inside a Pipeline:** same reason as V2: SMOTE must only touch the training part of each fold.
**Note:** it uses the best architecture / `alpha` from V3, and the same fixed-epoch training as V2.

In [ ]:
smote_mlp = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.5, random_state=SEED)),
    ('mlp',   make_mlp(early_stopping=False, max_iter=FIXED_EPOCHS, **best)),
])
evaluate_cv('V4', 'SMOTE (in-fold) + MLP, best V3 settings', smote_mlp, X_train, y_train, X_test)

### Block 8: V5 ablation, removing the "incomplete application" shortcut
**What:** drop the `pwsrc_*` columns (prevailing-wage source, including `pwsrc_Unknown` = source not recorded) and retrain with the V3 settings.
**Why:** the prevailing wage source is missing far more often for denied applications than for approved ones. That information is on the form at filing time (so it is not leakage), but a model could lean on it as a shortcut instead of learning the real drivers. The PR-AUC drop from V3 to V5 measures how much of the performance comes from this shortcut.
**Note:** V5 is a *diagnostic*, so it is not a candidate for the final model.

In [ ]:
pwsrc_cols = [c for c in X_train.columns if c.startswith('pwsrc_')]
print('Removed columns:', pwsrc_cols)
evaluate_cv('V5', 'Ablation: V3 without pwsrc_* columns', make_mlp(**best),
            X_train.drop(columns=pwsrc_cols), y_train, X_test.drop(columns=pwsrc_cols))

## 4. Comparison of the varieties

### Block 9: Comparison table and chart

In [ ]:
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(MODEL_DIR, 'cv_varieties.csv'), index=False)
show = comparison[['variety', 'description', 'pr_auc_mean', 'pr_auc_std', 'roc_auc_mean', 'f1_mean',
                   'recall_mean', 'precision_mean', 'balanced_acc_mean', 'accuracy_mean']]
display(show.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(comparison))

# Left: ranking quality. The red line is the PR-AUC of a random ranking (= the denied rate).
axes[0].bar(x, comparison['pr_auc_mean'], yerr=comparison['pr_auc_std'], capsize=4, color='#2980b9')
axes[0].axhline(y_train.mean(), color='#e74c3c', linestyle='--', label=f'random ranking = {y_train.mean():.3f}')
axes[0].set_xticks(x, comparison['variety'])
axes[0].set_ylabel('5-fold CV PR-AUC')
axes[0].set_title('Ranking quality (PR-AUC, mean +/- std)', fontweight='bold')
axes[0].legend()

# Right: recall / precision / accuracy at the default 0.5 threshold (shows the accuracy trap)
w = 0.25
for i, (m, c) in enumerate([('recall_mean', '#27ae60'), ('precision_mean', '#e67e22'), ('accuracy_mean', '#95a5a6')]):
    axes[1].bar(x + (i - 1) * w, comparison[m], w, label=m.replace('_mean', ''), color=c)
axes[1].set_xticks(x, comparison['variety'])
axes[1].set_title('Denied recall / precision vs accuracy (threshold 0.5)', fontweight='bold')
axes[1].legend()

plt.suptitle('Neural Network varieties: 5-fold CV on the train rows', fontweight='bold')
plt.savefig(os.path.join(MODEL_DIR, 'cv_varieties.png'), dpi=200, bbox_inches='tight')
plt.show()

### Block 10: Choose the best variety
**Rule (decided before looking at the test set):** the candidates are **V1-V4** (full 25-feature data).
- V0 is only the baseline;
- V5 is a diagnostic ablation.

Among V1-V4 the highest mean CV PR-AUC wins. If several are within one standard deviation of it, prefer in this order: the tuned model (V3), then plain (V1), then random oversampling (V2), then SMOTE (V4, synthetic data).

In [ ]:
candidates = comparison[comparison['variety'].isin(['V1', 'V2', 'V3', 'V4'])].sort_values('pr_auc_mean', ascending=False)
top = candidates.iloc[0]
preference = ['V3', 'V1', 'V2', 'V4']                # tie-break order (see the rule above)
close = candidates[candidates['pr_auc_mean'] >= top['pr_auc_mean'] - top['pr_auc_std']]
best_name = sorted(close['variety'], key=preference.index)[0]
best_estimator, best_X_train, best_X_test = fitted[best_name]
print(f'Highest CV PR-AUC: {top.variety} ({top.pr_auc_mean:.4f} +/- {top.pr_auc_std:.4f})')
print('Within one std of the top:', list(close['variety']))
print(f'Selected variety: {best_name} - {comparison.set_index("variety").loc[best_name, "description"]}')

## 5. Decision threshold

A probability model outputs P(denied). The default rule "denied if P >= 0.5" is arbitrary for imbalanced data. We choose the threshold that **maximises F1 for Denied**, using **out-of-fold** predictions on the train rows: every train row is predicted by a model that never saw it. The test set is *not* used for this choice.

### Block 11: Threshold tuning with out-of-fold predictions

In [ ]:
# Out-of-fold probabilities: 5 fits, each predicting the fold it did not train on
oof_proba = cross_val_predict(clone(best_estimator), best_X_train, y_train, cv=CV,
                              method='predict_proba', n_jobs=-1)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)   # F1 at every candidate threshold
best_i = int(np.argmax(f1s))
THRESHOLD = float(thr[best_i])                                                  # the chosen decision threshold

print(f'Out-of-fold PR-AUC: {average_precision_score(y_train, oof_proba):.4f}')
print(f'Best threshold = {THRESHOLD:.3f} -> OOF F1 {f1s[best_i]:.3f} (precision {prec[best_i]:.3f}, recall {rec[best_i]:.3f})')
default = (oof_proba >= 0.5).astype(int)
print(f'Default 0.5   -> OOF F1 {f1_score(y_train, default):.3f} (precision {precision_score(y_train, default, zero_division=0):.3f}, '
      f'recall {recall_score(y_train, default):.3f})')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(thr, prec[:-1], label='precision')
ax.plot(thr, rec[:-1], label='recall')
ax.plot(thr, f1s, label='F1', linewidth=2.5)
ax.axvline(THRESHOLD, color='black', linestyle='--', label=f'chosen threshold {THRESHOLD:.2f}')
ax.axvline(0.5, color='grey', linestyle=':', label='default 0.5')
ax.set_xlabel('Decision threshold on P(denied)')
ax.set_title(f'{best_name}: threshold vs precision / recall / F1 (out-of-fold, train rows)', fontweight='bold')
ax.legend()
plt.savefig(os.path.join(MODEL_DIR, 'threshold_tuning.png'), dpi=200, bbox_inches='tight')
plt.show()

## 6. Final evaluation on the test set (used once)

### Block 12: Fit on all train rows, then evaluate on the untouched test rows

In [ ]:
# Fit a fresh copy on ALL train rows (the CV only ever used 80% at a time)
final_model = clone(best_estimator).fit(best_X_train, y_train)
mlp = final_model.named_steps['mlp'] if hasattr(final_model, 'named_steps') else final_model   # the network itself

test_proba = final_model.predict_proba(best_X_test)[:, 1]
test_pred  = (test_proba >= THRESHOLD).astype(int)          # apply the threshold chosen on the train rows

n_weights = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
test_metrics = {
    'model': 'Neural Network (MLP)', 'variety': best_name,
    'architecture': f'{best_X_train.shape[1]} -> {" -> ".join(map(str, mlp.hidden_layer_sizes))} -> 1',
    'trainable_parameters': int(n_weights),
    'epochs_trained': int(mlp.n_iter_),
    'alpha': mlp.alpha,
    'threshold': round(THRESHOLD, 4),
    'cv_pr_auc_mean': round(float(comparison.set_index('variety').loc[best_name, 'pr_auc_mean']), 4),
    'cv_pr_auc_std':  round(float(comparison.set_index('variety').loc[best_name, 'pr_auc_std']), 4),
    'test_pr_auc':  round(average_precision_score(y_test, test_proba), 4),
    'test_roc_auc': round(roc_auc_score(y_test, test_proba), 4),
    'test_f1_denied':        round(f1_score(y_test, test_pred), 4),
    'test_precision_denied': round(precision_score(y_test, test_pred, zero_division=0), 4),
    'test_recall_denied':    round(recall_score(y_test, test_pred), 4),
    'test_balanced_accuracy': round(balanced_accuracy_score(y_test, test_pred), 4),
    'test_accuracy': round(accuracy_score(y_test, test_pred), 4),
}
with open(os.path.join(MODEL_DIR, 'test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(json.dumps(test_metrics, indent=2))
print()
print(classification_report(y_test, test_pred, target_names=['Approved', 'Denied'], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

# 1) Confusion matrix at the chosen threshold
ConfusionMatrixDisplay(confusion_matrix(y_test, test_pred), display_labels=['Approved', 'Denied']).plot(
    ax=axes[0], cmap='Blues', values_format=',', colorbar=False)
axes[0].set_title(f'Confusion matrix (test, threshold {THRESHOLD:.2f})', fontweight='bold')

# 2) Precision-Recall curve (the dot marks the chosen threshold)
p, r, _ = precision_recall_curve(y_test, test_proba)
axes[1].plot(r, p, color='#2980b9', linewidth=2, label=f"MLP (PR-AUC {test_metrics['test_pr_auc']:.3f})")
axes[1].axhline(y_test.mean(), color='#e74c3c', linestyle='--', label=f'random ({y_test.mean():.3f})')
axes[1].scatter([test_metrics['test_recall_denied']], [test_metrics['test_precision_denied']],
                color='black', zorder=5, label='chosen threshold')
axes[1].set_xlabel('Recall (Denied)'); axes[1].set_ylabel('Precision (Denied)')
axes[1].set_title('Precision-Recall curve (test)', fontweight='bold')
axes[1].legend()

# 3) ROC curve
fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[2].plot(fpr, tpr, color='#8e44ad', linewidth=2, label=f"MLP (ROC-AUC {test_metrics['test_roc_auc']:.3f})")
axes[2].plot([0, 1], [0, 1], color='grey', linestyle='--', label='random')
axes[2].set_xlabel('False positive rate'); axes[2].set_ylabel('True positive rate')
axes[2].set_title('ROC curve (test)', fontweight='bold')
axes[2].legend()

plt.savefig(os.path.join(MODEL_DIR, 'test_evaluation.png'), dpi=200, bbox_inches='tight')
plt.show()

### Block 12b: Training curve
A healthy curve falls quickly and then flattens. If a validation score is available (early-stopped varieties) it should flatten too; if it starts to fall while the loss keeps dropping, the network is over-fitting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(mlp.loss_curve_, label='training loss (log-loss)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Log-loss')
if getattr(mlp, 'validation_scores_', None) is not None:      # only exists when early_stopping=True
    ax2 = ax.twinx()
    ax2.plot(mlp.validation_scores_, color='#e67e22', label='internal validation accuracy')
    ax2.set_ylabel('Validation accuracy')
    ax2.legend(loc='center right')
ax.set_title(f'{best_name}: training curve ({mlp.n_iter_} epochs)', fontweight='bold')
ax.legend(loc='upper right')
plt.savefig(os.path.join(MODEL_DIR, 'training_curve.png'), dpi=200, bbox_inches='tight')
plt.show()

## 7. Interpretation: what drives denial?

### Block 13: Permutation importance
A neural network has no simple coefficients. Instead we ask: **if we randomly shuffle one feature (breaking its link with the target), how much does PR-AUC drop?** A big drop means the model relies on that feature; a drop near 0 means it barely matters.

We run it on a random 20,000-row sample of the test rows for speed. This is only an *explanation*: no modelling decision is taken from it, so the test set stays clean.
Correlated features share importance, so treat the ranking as a guide, not an exact truth.

In [ ]:
rng = np.random.RandomState(SEED)
n = min(20000, len(best_X_test))
idx = rng.choice(len(best_X_test), size=n, replace=False)
Xs, ys = best_X_test.iloc[idx], y_test.iloc[idx]

perm = permutation_importance(final_model, Xs, ys, scoring='average_precision',
                              n_repeats=5, random_state=SEED, n_jobs=-1)
imp = (pd.DataFrame({'feature': best_X_test.columns, 'pr_auc_drop': perm.importances_mean, 'std': perm.importances_std})
         .sort_values('pr_auc_drop', ascending=False))
imp.to_csv(os.path.join(MODEL_DIR, 'permutation_importance.csv'), index=False)

top15 = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15['feature'], top15['pr_auc_drop'], xerr=top15['std'], color='#2980b9', capsize=3)
ax.set_xlabel('Drop in PR-AUC when the feature is shuffled')
ax.set_title('Top 15 features by permutation importance', fontweight='bold')
plt.savefig(os.path.join(MODEL_DIR, 'permutation_importance.png'), dpi=200, bbox_inches='tight')
plt.show()
imp.head(10).round(4)

### Block 14: Save the model
The fitted model, the threshold and the feature list are saved, so the group comparison (and the demo) can reload the model without retraining.
In Colab, download the `results/` folder (or copy it to Google Drive) before the session ends.

In [ ]:
joblib.dump({'model': final_model, 'threshold': THRESHOLD, 'features': list(best_X_train.columns),
             'variety': best_name}, os.path.join(MODEL_DIR, 'model.joblib'))

# Reload and verify that the saved model reproduces exactly the same test predictions
reloaded = joblib.load(os.path.join(MODEL_DIR, 'model.joblib'))
check = (reloaded['model'].predict_proba(best_X_test[reloaded['features']])[:, 1] >= reloaded['threshold']).astype(int)
assert (check == test_pred).all()
print('Saved and verified:', sorted(os.listdir(MODEL_DIR)))

## 8. Conclusion (fill in after the full run)

### Results of the varieties (5-fold stratified CV on the train rows)

| Variety | What changed | CV PR-AUC (mean ± std) | Recall (Denied) | Precision (Denied) | Accuracy |
|---|---|---|---|---|---|
| V0 | Always approve (baseline) | | | | |
| V1 | Plain MLP (64, 32) | | | | |
| V2 | RandomOverSampler in-fold | | | | |
| V3 | GridSearchCV best | | | | |
| V4 | SMOTE in-fold + best V3 | | | | |
| V5 | Ablation: no `pwsrc_*` | | | | |

*(Copy the numbers from the table in Block 9. Recall, precision and accuracy use the default 0.5 threshold.)*

### Final model on the test set (from Block 12)
PR-AUC = `...`, ROC-AUC = `...`, F1 (Denied) = `...`, precision = `...`, recall = `...` at threshold `...`.

### Points to discuss
1. **Accuracy trap:** compare V0 / V1 accuracy with their recall for Denied.
2. **Resampling vs ranking:** did V2 / V4 change PR-AUC, or only recall / precision at 0.5?
3. **Tuning:** which architecture won in Block 6? Is there a gap between `train_pr_auc` and `cv_pr_auc` (over-fitting)?
4. **Shortcut check:** how much PR-AUC was lost in V5 (no `pwsrc_*`)?
5. **Neural Network vs Logistic Regression:** did the hidden layers find non-linear patterns that the linear model missed (compare PR-AUC with the group's Logistic Regression)?
6. **Limitations:** weaker interpretability, run-to-run variation from the random seed, and the fact that the data alone may limit how well denial can be predicted.